In [ ]:
import numpy as np
from freshflow.loader import load_sales
from freshflow.metrics import build_metrics
from freshflow.pipeline import run_pipeline
from freshflow.stats import OUTLET_NAMES, PRODUCT_NAMES, describe, ranking
from freshflow.validation import column

results = run_pipeline("../data/raw/freshflow_sales.csv", "../data/processed")
clean = load_sales("../data/processed/freshflow_clean.csv", expected_rows=None)
metrics = build_metrics(clean)
summary = results["summary"]
outlet = results["groups"]["outlet"]
product = results["groups"]["product"]
bands = results["bands"]
daily = results["daily_revenue"]

In [ ]:
print("Q1 outlet, highest net revenue:", OUTLET_NAMES[ranking(outlet, "revenue")[0]])
print("Q1 outlet, highest gross profit:", OUTLET_NAMES[ranking(outlet, "profit")[0]])
print("Q2 product, most units:", PRODUCT_NAMES[ranking(product, "units")[0]])
print("Q2 product, most gross profit:", PRODUCT_NAMES[ranking(product, "profit")[0]])

In [ ]:
print("Q3 margin, 0% discount band:", round(bands["margin_pct"][0], 2))
print("Q3 margin, >15 to 20% band:", round(bands["margin_pct"][4], 2))
print("Q3 average quantity per band:", np.round(bands["mean_quantity"], 2))

print("Q5 highest revenue day:", daily.argmax() + 1)
print("Q5 lowest revenue day:", daily.argmin() + 1)

revenue_stats = describe(metrics["net_revenue"])
print("Q6 mean transaction value:", round(revenue_stats["mean"], 2))
print("Q6 median transaction value:", round(revenue_stats["median"], 2))

In [ ]:
print("Q4 rows excluded as invalid:", results["excluded_rows"])
print("Q7 rows in exception register:", results["anomaly_rows"])

for row in results["scenarios"]:
    if row["response"] == 0.0:
        print(f"Q8 discount {row['rate']:.0%}: profit change {row['profit_change']:,.0f} ({row['profit_change_pct']:.2f}%)")

In [ ]:
expected = {
    1: {"net": 73.8625, "profit": 33.9125, "margin": 45.913},
    2: {"net": 50.81, "profit": 18.25, "margin": 35.918},
    3: {"net": 533.88, "profit": 305.68, "margin": 57.256},
}

for row_number, values in expected.items():
    index = row_number - 1
    np.testing.assert_allclose(metrics["net_revenue"][index], values["net"])
    np.testing.assert_allclose(metrics["gross_profit"][index], values["profit"])
    np.testing.assert_allclose(metrics["margin_pct"][index], values["margin"], rtol=1e-4)

print("Hand calculation matches NumPy")

In [ ]:
recommendations = [
    {
        "finding": "Margin falls from 47.04% to 34.43% as discounts rise, with no quantity lift",
        "action": "Test lower discounts (15% and above) at one outlet",
        "metric": "Gross margin %, average quantity per transaction",
        "limitation": "Association only, discounts were not assigned randomly",
        "next check": "Does quantity stay level when deep discounts are removed?",
    },
    {
        "finding": "Outlet C has 1,267 transactions against 1,981 at Outlet A, with the same basket value and margin",
        "action": "Investigate foot traffic, opening hours and staffing at Outlet C",
        "metric": "Daily transactions per outlet",
        "limitation": "No footfall or staffing data in the file",
        "next check": "What explains the transaction gap?",
    },
    {
        "finding": "15 invalid rows, 157 flagged rows, five 90% discounts and five zero-stock sales",
        "action": "Add point-of-sale input rules and review the flagged exceptions",
        "metric": "Invalid row rate (now 0.3%), flagged exception count",
        "limitation": "The transaction file cannot show which record is wrong",
        "next check": "Were the 90% discounts approved promotions or entry errors?",
    },
]

for number, item in enumerate(recommendations, start=1):
    print(f"Recommendation {number}")
    for key, value in item.items():
        print(f"  {key}: {value}")

In [ ]:
from freshflow.pipeline import SCENARIO_RATES

revenue = daily
quantity = column(clean, "quantity")
price = column(clean, "unit_price")
cost = column(clean, "unit_cost")
outlet_codes = column(clean, "outlet_id")
net = metrics["net_revenue"]

rates = np.linspace(0, 0.20, 5)
np.testing.assert_allclose(rates, SCENARIO_RATES)
print("linspace:", rates)
print("shares memory with asarray:", np.shares_memory(np.asarray(clean), clean))
print("eye times vector:", np.eye(3) @ np.array([1.0, 2.0, 3.0]))

blocks = revenue.reshape(5, 6)
print("6-day block totals:", np.round(blocks.sum(axis=1)))
print("transpose shape:", blocks.T.shape)
print("ravel is a view:", np.shares_memory(blocks.ravel(), revenue))
print("flatten is a copy:", not np.shares_memory(blocks.flatten(), revenue))

column_form = np.expand_dims(revenue, axis=1)
print("expand_dims and squeeze:", column_form.shape, np.squeeze(column_form).shape)

In [ ]:
print("10-day averages:", [round(part.mean()) for part in np.split(revenue, 3)])
print("array_split sizes:", [len(part) for part in np.array_split(revenue, 4)])

valid_net = net[~np.isnan(net)]
print("Largest 3 transactions:", np.sort(valid_net)[-3:])

has_price = ~np.isnan(price)
print("Weighted average price:", round(np.average(price[has_price], weights=quantity[has_price]), 2))
print("Plain average price:", round(price[has_price].mean(), 2))

rate = column(clean, "discount_rate")
print("Max discount, clipped copy (demo only):", np.clip(rate, 0, 0.2).max(), "| real max:", rate.max())

In [ ]:
from freshflow.stats import skewness

print("std by hand:", np.sqrt(np.var(valid_net, ddof=1)), "| np.std:", valid_net.std(ddof=1))
print("skewness before log1p:", round(skewness(valid_net), 3))
print("skewness after log1p:", round(skewness(np.log1p(valid_net)), 3))

has_profit = ~np.isnan(metrics["gross_profit"])
profit = metrics["gross_profit"][has_profit]
print("Profit from profitable rows:", round(np.maximum(profit, 0).sum(), 2))
print("Loss from loss-making rows:", round(np.minimum(profit, 0).sum(), 2))

In [ ]:
q = quantity[has_profit]
p = price[has_profit]
c = cost[has_profit]

print("matmul gross sales:", round(q @ p, 2), "| metrics:", round(metrics["gross_sales"][has_profit].sum(), 2))
print("einsum cogs:", round(np.einsum("i,i->", q, c), 2), "| metrics:", round(metrics["cogs"][has_profit].sum(), 2))

one_hot = outlet_codes[:, None] == np.array([1, 2, 3])
print("Revenue by outlet through matmul:", np.round(np.nan_to_num(net) @ one_hot))

start = np.datetime64("2026-11-01")
dates = start + np.arange(30).astype("timedelta64[D]")
print("Assumed dates (demo only):", dates[0], "to", dates[-1], "| span:", dates[-1] - dates[0])